<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/data_wrangling_assignment_answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
This is a great way to streamline your data cleaning process. Building this as a class makes your workflow modular and reusable across different Colab notebooks.

Here is a comprehensive Python class designed for Google Colab.

### The `DataInspector` Class

```python
import pandas as pd
import io
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

class DataInspector:
    def __init__(self):
        self.df = None

    def upload_data(self):
        """Uploads a CSV file and initializes the dataframe."""
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]))
        print(f"\n✅ File '{file_name}' loaded successfully!")

    def get_summary(self):
        """Displays basic stats and the top 20 rows."""
        if self.df is None: return print("Error: No data loaded.")

        num_cols = self.df.select_dtypes(include=['number']).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=['number']).columns.tolist()

        print(f"--- Data Summary ---")
        print(f"Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}")
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        print(f"\n--- Top 20 Rows ---")
        display(self.df.head(20))

    def show_missing_data(self):
        """Filters and displays only rows that contain at least one NaN value."""
        missing_rows = self.df[self.df.isnull().any(axis=1)]
        if missing_rows.empty:
            print("✨ No missing data found!")
        else:
            print(f"Rows with missing values ({len(missing_rows)} found):")
            display(missing_rows)

    def delete_rows(self):
        """Deletes a specified number of rows from the top and updates the dataframe."""
        try:
            n = int(input("Enter the number of rows to delete from the top: "))
            self.df = self.df.iloc[n:].reset_index(drop=True)
            print(f"🗑️ Deleted {n} rows. New row count: {len(self.df)}")
        except ValueError:
            print("Invalid input. Please enter an integer.")

    def column_details(self):
        """Shows unique categories for text or ranges for numbers."""
        for col in self.df.columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                col_range = self.df[col].max() - self.df[col].min()
                print(f"🔹 {col} (Numeric): Range = {col_range} [{self.df[col].min()} to {self.df[col].max()}]")
            else:
                unique_vals = self.df[col].unique()
                print(f"🔸 {col} (Categorical): {len(unique_vals)} classes -> {unique_vals[:5]}...")

    def plot_numerical(self, column_name):
        """Generates a Boxplot (for outliers) and a Scatter plot."""
        if column_name not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[column_name]):
            print("Error: Please provide a valid numerical column name.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Boxplot for Outliers
        sns.boxplot(x=self.df[column_name], ax=axes[0], color='skyblue')
        axes[0].set_title(f'Outlier Detection: {column_name}')

        # Scatter Plot (vs Index to see distribution)
        sns.scatterplot(x=self.df.index, y=self.df[column_name], ax=axes[1], alpha=0.5)
        axes[1].set_title(f'Scatter Distribution: {column_name}')

        plt.show()

# Initialize the inspector
inspector = DataInspector()

```

---

### How to use it in your Colab cells:

1. **Initialize and Upload:**
```python
inspector.upload_data()

```


2. **Get the breakdown:**
```python
inspector.get_summary()

```


3. **Check for gaps:**
```python
inspector.show_missing_data()

```


4. **Clean and Inspect Columns:**
```python
inspector.delete_rows()
inspector.column_details()

```


5. **Visualize:**
```python
# Replace 'Age' with a numerical column from your CSV
inspector.plot_numerical('Age')

```



### Key Features Explained

* **Outlier Detection:** The boxplot uses the Interquartile Range ($IQR$) method visually. Points outside the "whiskers" are statistically considered outliers.
* **Dynamic Data Handling:** By using `self.df = self.df.iloc[n:]`, the class ensures that once you delete rows, they are gone from the instance's memory for all subsequent method calls.
* **Categorical Logic:** Since some datasets have hundreds of categories, the `column_details` method shows a preview of the first 5 to prevent flooding your screen.

Would you like me to add a method that automatically fills those missing values with the column mean or median instead of just viewing them?